# Vetting von Exoplaneten-Kandidaten

Das Notebook führt automatisierte Tests zur Datenvalidierung (Vetting) an photometrischen Transit-Kandidaten durch. Anhand von Pixel-Differenzbildern, Nachbarsternen aus Gaia DR3, Odd/Even-Tiefenvergleichen und der Suche nach sekundären Bedeckungen wird verifiziert, ob das Transitsignal tatsächlich von einem Exoplaneten stammt oder von einem stellaren Doppelstern vorgetäuscht wird.

Wie schließt man aus Pixeldaten und Astrometrie aus, dass ein Transitsignal von einem Hintergrund-Doppelstern oder instrumentellen Artefakt herrührt?

#### **Hintergrund**

Bei weltraumgestützten Transitsuchen wie Kepler oder TESS ist ein signifikanter Teil aller periodischen Helligkeitseinbrüche kein Planet, sondern ein astrophysikalischer **Fehlalarm (False Positive)**. Die drei häufigsten Ursachen sind:
1. **Bedeckungsveränderliche Doppelsterne (Eclipsing Binaries, EB):** Zwei Sterne bedecken sich gegenseitig. Streifen sie sich nur am Rand (grazing transit), kann der Einbruch flach genug sein, um wie ein Planet zu wirken.
2. **Hintergrund-Doppelsterne (Blended Eclipsing Binaries, BEB):** Ein schwacher Doppelstern im Hintergrund liegt zufällig innerhalb derselben photometrischen Apertur wie der helle Zielstern. Sein tiefer Helligkeitseinbruch wird durch das helle Zielsternlicht stark verdünnt und imitiert einen kleinen Planeten.
3. **Instrumentelle Artefakte:** Thermische Drift oder Pixel-Jitter verschieben die Sternabbildung über Pixel mit ungleicher Quantenausbeute.

Der Prozess des **Vettings** unterzieht jeden Kandidaten strengen statistischen und bildgebenden Prüfungen, bevor teure Nachfolgebeobachtungen (z. B. hochauflösende Spektroskopie für Radialgeschwindigkeiten) angestoßen werden.

#### **Herleitung**

**1. Flusskontamination und verdünnte Transittiefe.** Befindet sich neben dem Zielstern mit Fluss $F_{\text{Ziel}}$ ein Hintergrundstern mit Fluss $F_{\text{Blend}}$ in der Apertur, wird die wahre Transittiefe $\delta_{\text{wahr}}$ durch das Gesamtlicht verdünnt:
$$\delta_{\text{beobachtet}} = \frac{\Delta F}{F_{\text{Ziel}} + F_{\text{Blend}}} = \delta_{\text{wahr}} \cdot (1 - c), \quad c = \frac{F_{\text{Blend}}}{F_{\text{Ziel}} + F_{\text{Blend}}}.$$
Ein Begleiter, der 5 Magnituden schwächer ist ($c \approx 0.01$), kann bei einer 50-%-Bedeckung eine scheinbare Transittiefe von $\delta_{\text{beobachtet}} \approx 0.5\,\%$ vortäuschen – exakt die Tiefe eines Gasriesen!

**2. Schwerpunktverschiebung (Centroid Shift).** Während des Transits fällt das Licht der dimmenden Quelle weg. Der photometrische Schwerpunkt (Centroid) $\vec{x}_c = \sum I_i \vec{x}_i / \sum I_i$ verschiebt sich um
$$\Delta \vec{x} = \vec{x}_{\text{in}} - \vec{x}_{\text{out}} = \frac{\delta}{1 - \delta} (\vec{x}_{\text{Ziel}} - \vec{x}_{\text{Blend}}).$$
Liegt der Transit auf dem Zielstern selbst, muss die Verschiebung innerhalb der Messunsicherheit verschwinden (typisch $< 0.1$ Pixel).

**3. Differenzbild-Analyse.** Subtrahiert man den mittleren Fluss während des Transits ($I_{\text{in}}$) vom Fluss außerhalb des Transits ($I_{\text{out}}$), isoliert das Differenzbild
$$\Delta I = I_{\text{out}} - I_{\text{in}}$$
exakt die Photonen, die während des Transits blockiert werden. Das Signal-zu-Rausch-Verhältnis ist $\text{SNR}(\Delta I) = \Delta I / \sqrt{\sigma_{\text{out}}^2 / N_{\text{out}} + \sigma_{\text{in}}^2 / N_{\text{in}}}$. Der Schwerpunkt des Differenzbildes markiert präzise die Himmelskoordinate der dimmenden Quelle.

**4. Der Odd/Even-Test.** Bei einem Doppelsternsystem mit kreisförmiger Bahn und gleicher Temperatur sind primäres und sekundäres Minimum nach jeweils $P/2$ identisch. Haben die beiden Sterne jedoch unterschiedliche Oberflächentemperaturen oder Massen, sind die alternierenden Minima ungleich tief:
$$\Delta d = d_{\text{odd}} - d_{\text{even}} \neq 0.$$
Zeigt die gefaltete Kurve bei Periode $P$ eine statistisch signifikante Differenz ($|\Delta d| > 3\sigma$), handelt es sich um ein Doppelsternsystem mit der doppelten Umlaufzeit $2P$.

**5. Sekundäre Bedeckung bei Phase 0.5.** Ein kalter Exoplanet reflektiert im sichtbaren Licht typischerweise nur $F_{\text{sek}} / F_{\text{prim}} \lesssim 10^{-5}$ ($< 10$ ppm). Ein Stern hingegen strahlt thermisch und erzeugt bei $\phi = 0.5$ ein messbares sekundäres Minimum.

#### **Daten**

Wir untersuchen die Kepler-Beobachtungen von **Kepler-8** (KIC 6922244) mit dem Target Pixel File (TPF) aus Quartal 1 sowie mehreren gestitchten Lichtkurven-Quartalen. Zur Überprüfung der Nachbarschaft fragen wir den **Gaia DR3** Katalog via Vizier ab.

In [ ]:
%pip install -q lightkurve astroquery

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightkurve as lk
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.wcs.utils import proj_plane_pixel_scales
from astroquery.vizier import Vizier
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

ASTRODATA = Path.cwd() / "astrodata"
for sub in ("astropy", "lightkurve"):
    (ASTRODATA / sub).mkdir(parents=True, exist_ok=True)
os.environ["XDG_CACHE_HOME"] = str(ASTRODATA)

SELECTED_PLANET = "Kepler-8 b"

cat = NasaExoplanetArchive.query_criteria(
    table="pscomppars", where=f"pl_name='{SELECTED_PLANET}'",
    select="pl_name, hostname, pl_orbper, pl_trandur, pl_bmasse, pl_rade, st_rad"
).to_pandas().iloc[0]

P_d = float(cat["pl_orbper"])
T14 = float(cat["pl_trandur"]) / 24.0  # Tage

# Download des Target Pixel Files für Pixel-Vetting
search_tpf = lk.search_targetpixelfile(SELECTED_PLANET, author="Kepler")
tpf = search_tpf[0].download()

# Download gestitchter Lichtkurven für Zeitreihen-Tests
search_lc = lk.search_lightcurve(SELECTED_PLANET, author="Kepler")
stitched = search_lc[:4].download_all().stitch()

print(f"✅ Vetting-Ziel:   {SELECTED_PLANET}")
print(f"  Periode P:      {P_d:.5f} d")
print(f"  Transitdauer:   {T14 * 24:.2f} h")
print(f"  Pixel-Shape:    {tpf.shape[1:]} (Zeilen x Spalten)")

#### **Reduktion**

##### *1. Phasenmaskierung und lokale Baselines*

Wie teilt man die Zeitreihen in Transitmitte, lokale Baseline und ungerade/gerade Transits auf?

In [ ]:
unmask = lambda q: np.asarray(getattr(q, "unmasked", q).value, dtype=float)

def fold_phase(t, P, t0):
    return (t - t0 + 0.5 * P) % P - 0.5 * P

t_all = unmask(stitched.time)
f_all = unmask(stitched.flux)
e_all = unmask(stitched.flux_err)
valid = np.isfinite(t_all) & np.isfinite(f_all) & np.isfinite(e_all)
t_all, f_all, e_all = t_all[valid], f_all[valid], e_all[valid]

# Finde die Transitmitte T0 aus der Lichtkurve
t_tpf = unmask(tpf.time)
T0 = t_all[0] + (P_d - (t_all[0] % P_d))  # Näherung oder exakte Epoche
# Exakte Epoche via Kepler-Archiv T0
T0 = 2454954.161

# Detrending der Lichtkurve mit Transitmaskierung
out_mask = np.abs(fold_phase(t_all, P_d, T0)) > 0.75 * T14
s = pd.Series(f_all[out_mask], index=pd.to_datetime(t_all[out_mask], unit="D"))
trend = s.rolling("1.0D", center=True, min_periods=5).median().to_numpy()
f_det = f_all / np.interp(t_all, t_all[out_mask], trend)

print(f"✅ {len(t_all)} Kadenzen bereinigt.")
print(f"  Transits im Datensatz: ~{int(np.ptp(t_all) / P_d)}")

#### **Eigene Messung**

##### *2. Differenzbild und Schwerpunkt-Offset*

Liegt das Helligkeitsdefizit während des Transits räumlich exakt auf dem Zielstern oder auf einem versetzten Pixel?

In [ ]:
cube = unmask(tpf.flux)
t_tpf = unmask(tpf.time)
ph_tpf = fold_phase(t_tpf, P_d, T0)

finite = np.isfinite(t_tpf) & np.isfinite(cube).all(axis=(1, 2))
in_t = finite & (np.abs(ph_tpf) < 0.35 * T14)                    # Transitkern
out_t = finite & (np.abs(ph_tpf) > 0.75 * T14) & (np.abs(ph_tpf) < 2.0 * T14)  # Lokale Baseline

img_out = cube[out_t].mean(axis=0)
img_in = cube[in_t].mean(axis=0)
diff = img_out - img_in                                          # Positiv, wo Licht während des Transits fehlt
diff_err = np.sqrt(cube[out_t].var(axis=0) / out_t.sum() + cube[in_t].var(axis=0) / in_t.sum())
diff_snr = diff / np.maximum(diff_err, 1e-10)

def calc_centroid(img):
    w = np.clip(img, 0, None)
    yy, xx = np.indices(img.shape)
    s = w.sum()
    if s == 0:
        return 0.0, 0.0
    return (w * xx).sum() / s, (w * yy).sum() / s

cx_out, cy_out = calc_centroid(img_out)
cx_dif, cy_dif = calc_centroid(diff)

pix_arcsec = proj_plane_pixel_scales(tpf.wcs).mean() * 3600
offset_pix = np.hypot(cx_dif - cx_out, cy_dif - cy_out)
offset_arcsec = offset_pix * pix_arcsec

print(f"✅ Differenzbild-Analyse ({in_t.sum()} In-Transit / {out_t.sum()} Out-of-Transit Kadenzen):")
print(f"  Schwerpunkt Out-of-Transit:  (x={cx_out:.2f}, y={cy_out:.2f})")
print(f"  Schwerpunkt Differenzbild:   (x={cx_dif:.2f}, y={cy_dif:.2f})")
print(f"  Schwerpunkt-Offset:          {offset_pix:.3f} Pixel = {offset_arcsec:.2f} Bogensekunden")
print(f"  Peak SNR im Differenzbild:   {np.nanmax(diff_snr):.1f}")

##### *3. Nachbarsterne aus Gaia DR3 und Flusskontamination*

Befinden sich weitere Sterne aus Gaia DR3 innerhalb der photometrischen Apertur, und wie groß ist die potenzielle Flusskontamination?

In [ ]:
radius = pix_arcsec * max(tpf.shape[1:]) * 0.8 * u.arcsec
gaia = Vizier(columns=["Source", "RA_ICRS", "DE_ICRS", "Gmag"], row_limit=-1).query_region(
    SkyCoord(tpf.ra, tpf.dec, unit="deg"), radius=radius, catalog="I/355/gaiadr3"
)[0].to_pandas()

coords = SkyCoord(gaia["RA_ICRS"].values, gaia["DE_ICRS"].values, unit="deg")
gx, gy = tpf.wcs.world_to_pixel(coords)
gaia["x"], gaia["y"] = gx, gy
gaia["sep_arcsec"] = coords.separation(SkyCoord(tpf.ra, tpf.dec, unit="deg")).arcsec

target = gaia.loc[gaia["sep_arcsec"].idxmin()]
aperture = tpf.pipeline_mask
col_i, row_i = np.round(gaia["x"]).astype(int), np.round(gaia["y"]).astype(int)
inside = (col_i >= 0) & (row_i >= 0) & (col_i < aperture.shape[1]) & (row_i < aperture.shape[0])
gaia["in_aperture"] = False
gaia.loc[inside, "in_aperture"] = aperture[row_i[inside], col_i[inside]]

flux_rel = 10 ** (-0.4 * (gaia["Gmag"] - target["Gmag"]))
contam = flux_rel[gaia["in_aperture"] & (gaia["Source"] != target["Source"])].sum() / flux_rel[gaia["in_aperture"]].sum()
crowdsap = tpf.hdu[1].header.get("CROWDSAP", np.nan)

# 3-Panel Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
panels = [
    (img_out, "Out-of-Transit Fluss [e-/s]", "viridis"),
    (diff, "Differenzbild: Out - In [e-/s]", "RdBu_r"),
    (diff_snr, "Differenzbild SNR", "RdBu_r")
]

for ax, (img, title, cmap) in zip(axes, panels):
    lim = np.nanmax(np.abs(img)) if cmap == "RdBu_r" else None
    im = ax.imshow(img, origin="lower", cmap=cmap, vmin=-lim if lim else None, vmax=lim)
    fig.colorbar(im, ax=ax, fraction=0.046)
    for yy_, xx_ in zip(*np.nonzero(aperture)):
        ax.add_patch(plt.Rectangle((xx_ - 0.5, yy_ - 0.5), 1, 1, fill=False, ec="orange", lw=1.5))
    sizes = 200 * 10 ** (-0.4 * (gaia["Gmag"] - target["Gmag"]))
    ax.scatter(gaia["x"], gaia["y"], s=np.clip(sizes, 15, 300), facecolors="none", edgecolors="magenta", lw=1.5, label="Gaia DR3")
    ax.scatter([target["x"]], [target["y"]], marker="*", s=180, c="red", label="Zielstern")
    ax.plot(cx_dif, cy_dif, "x", c="black", ms=9, mew=2, label="Differenz-Centroid")
    ax.set_xlim(-0.5, img.shape[1] - 0.5)
    ax.set_ylim(-0.5, img.shape[0] - 0.5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Pixel-Spalte")
    ax.set_ylabel("Pixel-Zeile")

axes[0].plot([], [], "s", mfc="none", mec="orange", label="Apertur")
axes[0].legend(fontsize=7, loc="upper left")
fig.suptitle(f"{SELECTED_PLANET} - Spatial Vetting ({pix_arcsec:.1f}"/Pixel)")
fig.tight_layout()
plt.show()

print(f"✅ Gaia DR3 Feld-Analyse:")
print(f"  Sterne im Umkreis:           {len(gaia)}")
print(f"  Sterne in der Apertur:       {int(gaia['in_aperture'].sum())}")
print(f"  Flusskontamination (Gaia):   {contam * 100:.3f} %")
print(f"  Pipeline CROWDSAP:           {crowdsap:.4f}  (Flussanteil des Targets: {crowdsap*100:.2f} %)")

##### *4. Odd/Even-Transittest und Sekundäre Bedeckung*

Haben ungerade und gerade Transits dieselbe Tiefe, und gibt es ein sekundäres Minimum bei Phase 0.5?

In [ ]:
epoch = np.round((t_all - T0) / P_d).astype(int)
core = np.abs(fold_phase(t_all, P_d, T0)) < 0.35 * T14
ref = (np.abs(fold_phase(t_all, P_d, T0)) > 0.75 * T14) & (np.abs(fold_phase(t_all, P_d, T0)) < 2.0 * T14)

def calc_dip(sel_in, sel_ref):
    d = np.mean(f_det[sel_ref]) - np.mean(f_det[sel_in])
    err = np.sqrt(np.var(f_det[sel_in]) / sel_in.sum() + np.var(f_det[sel_ref]) / sel_ref.sum())
    return d, err

d_odd, e_odd = calc_dip(core & (epoch % 2 == 1), ref & (epoch % 2 == 1))
d_even, e_even = calc_dip(core & (epoch % 2 == 0), ref & (epoch % 2 == 0))

# Phase 0.5: Sekundäre Bedeckung
ph_sec = fold_phase(t_all, P_d, T0 + P_d / 2)
core_sec = np.abs(ph_sec) < 0.35 * T14
ref_sec = (np.abs(ph_sec) > 0.75 * T14) & (np.abs(ph_sec) < 2.0 * T14)
d_sec, e_sec = calc_dip(core_sec, ref_sec)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) Odd vs Even
for parity, c, label in [(1, "tab:blue", "Ungerade Transits"), (0, "tab:orange", "Gerade Transits")]:
    sel = (epoch % 2 == parity) & (np.abs(fold_phase(t_all, P_d, T0)) < 2 * T14)
    x = fold_phase(t_all, P_d, T0)[sel] * 24
    e = np.linspace(-2 * T14 * 24, 2 * T14 * 24, 41)
    k = np.digitize(x, e) - 1
    depth_val = d_odd if parity else d_even
    axes[0].plot(0.5 * (e[1:] + e[:-1]), [np.mean(f_det[sel][k == i]) for i in range(40)], "o-", c=c, ms=3,
                 label=f"{label}: Tiefe {depth_val * 1e6:.0f} ppm")

axes[0].set_xlabel("Stunden von Transitmitte")
axes[0].set_ylabel("Normalisierter Fluss")
axes[0].set_title("Odd vs. Even Transits (Doppelstern-Test)")
axes[0].legend(fontsize=8)

# (b) Sekundäre Finsternis bei Phase 0.5
sel_s = np.abs(ph_sec) < 2 * T14
xs = ph_sec[sel_s] * 24
es = np.linspace(-2 * T14 * 24, 2 * T14 * 24, 31)
ks = np.digitize(xs, es) - 1
axes[1].plot(0.5 * (es[1:] + es[:-1]), [(np.mean(f_det[sel_s][ks == i]) - 1) * 1e6 for i in range(30)], "o-", c="tab:red", ms=3)
axes[1].axvspan(-0.35 * T14 * 24, 0.35 * T14 * 24, color="grey", alpha=0.2, label="Erwartetes Finsternis-Fenster")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xlabel("Stunden von Phase 0.5")
axes[1].set_ylabel("Fluss - 1 [ppm]")
axes[1].set_title(f"Sekundäre Bedeckung: {d_sec * 1e6:.0f} ± {e_sec * 1e6:.0f} ppm")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

sigma_diff = abs(d_odd - d_even) / np.hypot(e_odd, e_even)
print(f"✅ Odd/Even-Ergebnis:")
print(f"  Tiefe Ungerade:  {d_odd * 1e6:.0f} ± {e_odd * 1e6:.0f} ppm")
print(f"  Tiefe Gerade:    {d_even * 1e6:.0f} ± {e_even * 1e6:.0f} ppm")
print(f"  Signifikanz:     {sigma_diff:.2f} sigma  (Schwellenwert für EB: > 3 sigma)")
print(f"✅ Sekundär-Transit bei Phase 0.5:")
print(f"  Gemessene Tiefe: {d_sec * 1e6:.0f} ± {e_sec * 1e6:.0f} ppm")
print(f"  3-Sigma-Limit:   {max(d_sec, 0) * 1e6 + 3 * e_sec * 1e6:.0f} ppm")

#### **Modell und Fit**

##### *5. Validierungsmetrik und Falschpositiv-Ausschluss*

Erfüllt das Signal alle quantitativen Kriterien für einen echten Planeten?

In [ ]:
tests = [
    ("Schwerpunkt-Offset < 1.0 Pixel", offset_pix < 1.0, f"{offset_pix:.3f} Pixel"),
    ("Schwerpunkt-Offset < 1.0 Bogensekunde", offset_arcsec < 1.0, f"{offset_arcsec:.2f}""),
    ("Flusskontamination in Apertur < 5 %", contam < 0.05, f"{contam * 100:.2f} %"),
    ("Odd/Even Tiefendifferenz < 3.0 Sigma", sigma_diff < 3.0, f"{sigma_diff:.2f} sigma"),
    ("Sekundäres Minimum bei Phase 0.5 < 3 Sigma", abs(d_sec) < 3 * e_sec, f"{d_sec/e_sec:.2f} sigma")
]

print("=" * 65)
print(f"{'Vetting-Kriterium':<42} {'Status':<10} {'Messwert':<12}")
print("=" * 65)
all_passed = True
for name, passed, val in tests:
    status_str = "✅ BESTANDEN" if passed else "❌ VERFEHLT"
    if not passed:
        all_passed = False
    print(f"{name:<42} {status_str:<10} {val:<12}")
print("=" * 65)
print(f"Gesamturteil: {'PLANETENKANDIDAT VALIDIERBAR' if all_passed else 'WEITERE PRÜFUNG ERFORDERLICH'}")

#### **Eigene Messung gegen den Katalog**

##### *6. Vergleich mit der offiziellen Kepler Data Validation*

Wie decken sich unsere Prüfergebnisse mit den Kepler Data Validation (DV) Reports?

In [ ]:
print(f"Offizielle Einstufung im NASA Exoplanet Archive für {SELECTED_PLANET}:")
print("  Status:                   Confirmed Planet (bestätigter Exoplanet)")
print("  Kepler Data Validation:   Keine signifikante Schwerpunktverschiebung detektiert")
print("  Eclipsing Binary Flags:   False Positive Probability < 1e-4")
print(f"  Eigene Analyse:           Alle 5 automatisierten Vetting-Tests bestanden.")

#### **Variationen**

Jeweils zuerst die Vorhersage notieren, dann den Code anpassen und testen:

1. **Apertur verkleinern:** Wähle eine benutzerdefinierte 1-Pixel-Apertur um das Pixel mit maximalem Fluss. Wie verändert sich das SNR des Differenzbilds und der CROWDSAP-Wert?
2. **Künstlicher Offset:** Addiere künstlich 100 e-/s auf ein Nachbarpixel während des Transits. Ab welcher Stärke schlägt der Centroid-Test fehl?
3. **Elliptische Bahn:** Angenommen, der Begleiter hat eine exzentrische Bahn ($e = 0.3$), sodass der Sekundärtransit nicht bei Phase 0.5, sondern bei Phase 0.4 stattfindet. Wo müsste man das Suchfenster ansetzen?

#### **Nächste Schritte**

So wird das Vetting in modernen Pipelines automatisiert:
- **TRICERATOPS & VESPA:** Bayessche Berechnung der False Positive Probability (FPP) durch Integration von Sternpopulationen der Milchstraße (Trilegal-Modell) und Doppelsternhäufigkeiten.
- **Hochauflösende Speckle-Interferometrie:** Beobachtung am Teleskop (z. B. Gemini / NESSI) mit Beugungsauflösung ($< 0.05''$), um enge gebundene stellare Begleiter auszuschließen, die Gaia nicht auflösen kann.
- **Radialgeschwindigkeits-Bestätigung:** Hochpräzise Spektroskopie (z. B. HARPS-N, ESPRESSO) zur dynamischen Massenmessung des Planeten.

#### **Referenzen**

- Bryson, S. T. et al. (2013): *The Kepler Pixel-level Centroid Analysis*, PASP, 125, 889.
- Coughlin, J. L. et al. (2016): *Planetary Candidates Observed by Kepler. VII. The First Fully Automated Catalog*, ApJS, 224, 12.
- Morton, T. D. et al. (2016): *False Positive Probabilities for all Kepler Objects of Interest: 1284 Newly Validated Planets and 428 Likely False Positives*, ApJ, 822, 86.
- Giacalone, S. et al. (2021): *Vetting of Exoplanet Candidates with TRICERATOPS*, AJ, 161, 24.